# NDVI in FN Mangroves Before vs After Hurricane Melissa

This notebook extracts NDVI pixels within Forces of Nature (FN) mangrove polygons and compares:
- `HLS_masked_NDVI_2months_before_epsg3448_2025-08-21_to_2025-10-21.tif`
- `HLS_masked_NDVI_2months_after_epsg3448_2025-10-29_to_2025-12-29.tif`

Outputs include:
- Mangrove-only NDVI summary statistics
- Before vs after histogram
- NDVI change (`after - before`) histogram using paired valid pixels
- Saved figures (and optional pixel export) in `results/threats/ndvi/draft_processed_images`


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio.features import geometry_mask
import matplotlib.pyplot as plt

plt.style.use('default')
pd.set_option('display.max_columns', 100)


In [ ]:
def find_project_root(start: Path) -> Path:
    start = start.resolve()
    for p in [start] + list(start.parents):
        if (p / 'dphil_papers').exists():
            return p
    raise FileNotFoundError(f'Could not find project root from {start}')

ROOT = find_project_root(Path.cwd())

ndvi_before_path = ROOT / 'dphil_papers/dphil_paper_3/inputs/ndvi/HLS_masked_NDVI_2months_before_epsg3448_2025-08-21_to_2025-10-21.tif'
ndvi_after_path = ROOT / 'dphil_papers/dphil_paper_3/inputs/ndvi/HLS_masked_NDVI_2months_after_epsg3448_2025-10-29_to_2025-12-29.tif'
mangrove_fn_path = ROOT / 'dphil_papers/dphil_paper_3/inputs/forces_of_nature_mangroves/mangroves.shp'

output_dir = ROOT / 'dphil_papers/dphil_paper_3/results/threats/ndvi/draft_processed_images'
output_dir.mkdir(parents=True, exist_ok=True)

print('Working dir:', Path.cwd())
print('Project root:', ROOT)
for p in [ndvi_before_path, ndvi_after_path, mangrove_fn_path]:
    print(p.name, 'exists ->', p.exists())
print('Output dir:', output_dir)


# Control file exports while iterating
SAVE_PNGS = False


In [ ]:
with rasterio.open(ndvi_before_path) as src_b, rasterio.open(ndvi_after_path) as src_a:
    before_meta = {
        'crs': str(src_b.crs),
        'shape': (src_b.height, src_b.width),
        'count': src_b.count,
        'dtype': src_b.dtypes[0],
        'nodata': src_b.nodata,
        'bounds': src_b.bounds,
        'transform': src_b.transform,
    }
    after_meta = {
        'crs': str(src_a.crs),
        'shape': (src_a.height, src_a.width),
        'count': src_a.count,
        'dtype': src_a.dtypes[0],
        'nodata': src_a.nodata,
        'bounds': src_a.bounds,
        'transform': src_a.transform,
    }

same_grid = (
    before_meta['crs'] == after_meta['crs']
    and before_meta['shape'] == after_meta['shape']
    and before_meta['transform'] == after_meta['transform']
)

before_meta_print = {k: v for k, v in before_meta.items() if k != 'transform'}
after_meta_print = {k: v for k, v in after_meta.items() if k != 'transform'}

print('Before metadata:')
print(pd.Series(before_meta_print))
print('\nAfter metadata:')
print(pd.Series(after_meta_print))
print('\nSame grid:', same_grid)


In [ ]:
mangroves = gpd.read_file(mangrove_fn_path)

with rasterio.open(ndvi_before_path) as src:
    raster_crs = src.crs
    raster_bounds = src.bounds

if mangroves.crs != raster_crs:
    mangroves = mangroves.to_crs(raster_crs)

mangroves = mangroves[mangroves.geometry.notnull() & ~mangroves.geometry.is_empty].copy()

print('Mangrove CRS:', mangroves.crs)
print('Mangrove polygons:', len(mangroves))
print('Mangrove area (ha):', round(mangroves.geometry.area.sum() / 10_000, 2))


In [ ]:
if not same_grid:
    raise ValueError('Before/after NDVI rasters are not on the same grid. Reproject/resample first.')

with rasterio.open(ndvi_before_path) as src_b, rasterio.open(ndvi_after_path) as src_a:
    arr_before = src_b.read(1)
    arr_after = src_a.read(1)

    mangrove_mask = geometry_mask(
        [geom for geom in mangroves.geometry if geom is not None and not geom.is_empty],
        transform=src_b.transform,
        out_shape=(src_b.height, src_b.width),
        invert=True,
    )

    valid_before = np.isfinite(arr_before) & (arr_before >= -1.0) & (arr_before <= 1.0)
    valid_after = np.isfinite(arr_after) & (arr_after >= -1.0) & (arr_after <= 1.0)

    if src_b.nodata is not None and np.isfinite(src_b.nodata):
        valid_before &= arr_before != src_b.nodata
    if src_a.nodata is not None and np.isfinite(src_a.nodata):
        valid_after &= arr_after != src_a.nodata

    valid_before_m = valid_before & mangrove_mask
    valid_after_m = valid_after & mangrove_mask
    valid_paired_m = valid_before & valid_after & mangrove_mask

    ndvi_before_m = arr_before[valid_before_m]
    ndvi_after_m = arr_after[valid_after_m]

    ndvi_before_paired = arr_before[valid_paired_m]
    ndvi_after_paired = arr_after[valid_paired_m]
    ndvi_delta_paired = ndvi_after_paired - ndvi_before_paired

print('Mangrove pixels (before-valid):', f'{ndvi_before_m.size:,}')
print('Mangrove pixels (after-valid):', f'{ndvi_after_m.size:,}')
print('Mangrove pixels (paired valid):', f'{ndvi_delta_paired.size:,}')


In [ ]:
def summarize(vals: np.ndarray, label: str) -> dict:
    return {
        'series': label,
        'n': int(vals.size),
        'mean': float(np.mean(vals)),
        'median': float(np.median(vals)),
        'std': float(np.std(vals)),
        'min': float(np.min(vals)),
        'p05': float(np.percentile(vals, 5)),
        'p25': float(np.percentile(vals, 25)),
        'p75': float(np.percentile(vals, 75)),
        'p95': float(np.percentile(vals, 95)),
        'max': float(np.max(vals)),
    }

summary_df = pd.DataFrame([
    summarize(ndvi_before_m, 'before (mangrove pixels)'),
    summarize(ndvi_after_m, 'after (mangrove pixels)'),
    summarize(ndvi_delta_paired, 'delta after-before (paired mangrove pixels)'),
]).set_index('series').round(4)

summary_df


In [ ]:
# Impact indicators for mangrove NDVI change
# Mutually exclusive direction classes:
# down: delta < 0, same: delta == 0, up: delta > 0

impact_indicators = {
    'n_paired_pixels': int(ndvi_delta_paired.size),
    'mean_delta_ndvi': float(np.mean(ndvi_delta_paired)),
    'median_delta_ndvi': float(np.median(ndvi_delta_paired)),
    'p25_delta_ndvi': float(np.percentile(ndvi_delta_paired, 25)),
    'p75_delta_ndvi': float(np.percentile(ndvi_delta_paired, 75)),
    'pct_down_delta_lt_0': float(100.0 * np.mean(ndvi_delta_paired < 0.0)),
    'pct_same_delta_eq_0': float(100.0 * np.mean(ndvi_delta_paired == 0.0)),
    'pct_up_delta_gt_0': float(100.0 * np.mean(ndvi_delta_paired > 0.0)),
    'pct_moderate_decline_delta_lte_minus_0p05': float(100.0 * np.mean(ndvi_delta_paired <= -0.05)),
    'pct_severe_decline_delta_lte_minus_0p10': float(100.0 * np.mean(ndvi_delta_paired <= -0.10)),
    'pct_moderate_improve_delta_gte_0p05': float(100.0 * np.mean(ndvi_delta_paired >= 0.05)),
    'pct_strong_improve_delta_gte_0p10': float(100.0 * np.mean(ndvi_delta_paired >= 0.10)),
}

impact_df = pd.DataFrame([impact_indicators]).T.rename(columns={0: 'value'})
impact_df.index.name = 'indicator'
impact_df['value'] = impact_df['value'].astype(float)
impact_df


In [ ]:
# Save impact indicators for reporting
impact_csv = output_dir / 'ndvi_mangroves_impact_indicators.csv'
impact_df.reset_index().to_csv(impact_csv, index=False)
print('Saved:', impact_csv)


In [ ]:
# Histogram: NDVI before vs after in mangrove pixels
fig, ax = plt.subplots(figsize=(9, 5), constrained_layout=True)

low = min(np.percentile(ndvi_before_m, 0.5), np.percentile(ndvi_after_m, 0.5))
high = max(np.percentile(ndvi_before_m, 99.5), np.percentile(ndvi_after_m, 99.5))
bins = np.linspace(low, high, 90)

ax.hist(ndvi_before_m, bins=bins, alpha=0.50, density=True, label='Before event')
ax.hist(ndvi_after_m, bins=bins, alpha=0.50, density=True, label='After event')
ax.set_xlabel('NDVI')
ax.set_ylabel('Density')
ax.set_title('Mangrove NDVI Distribution: Before vs After Hurricane Melissa')
ax.legend(loc='upper left')

before_after_hist_png = output_dir / 'ndvi_mangroves_before_after_hist.png'
if SAVE_PNGS:
    fig.savefig(before_after_hist_png, dpi=300)
    print('Saved:', before_after_hist_png)
else:
    print('PNG export skipped (SAVE_PNGS=False):', before_after_hist_png)
plt.show()

In [ ]:
# Histogram: NDVI change in paired valid mangrove pixels
fig, ax = plt.subplots(figsize=(9, 5), constrained_layout=True)

q_low, q_high = np.percentile(ndvi_delta_paired, [0.5, 99.5])
bins = np.linspace(q_low, q_high, 90)

ax.hist(ndvi_delta_paired, bins=bins, alpha=0.75, color='#3366cc', density=True)
ax.axvline(0, color='black', linestyle='--', linewidth=1)
ax.set_xlabel('NDVI change (after - before)')
ax.set_ylabel('Density')
ax.set_title('Mangrove NDVI Change Distribution (Paired Pixels)')

delta_hist_png = output_dir / 'ndvi_mangroves_change_hist.png'
if SAVE_PNGS:
    fig.savefig(delta_hist_png, dpi=300)
    print('Saved:', delta_hist_png)
else:
    print('PNG export skipped (SAVE_PNGS=False):', delta_hist_png)
plt.show()

In [ ]:
# Optional: export paired mangrove NDVI values for downstream analysis
paired_df = pd.DataFrame({
    'ndvi_before': ndvi_before_paired,
    'ndvi_after': ndvi_after_paired,
    'ndvi_change_after_minus_before': ndvi_delta_paired,
})

csv_out = output_dir / 'ndvi_mangroves_paired_pixels_before_after.csv'
paired_df.to_csv(csv_out, index=False)

print('Saved:', csv_out)
paired_df.head()


## Notes
- NDVI is expected in `[-1, 1]`; values outside this range are filtered out.
- `before`/`after` histograms include all valid mangrove pixels per raster.
- `delta` uses only paired pixels valid in both rasters.


In [ ]:
# Combined 3-panel figure for sharing
from matplotlib.gridspec import GridSpec

combined_png = output_dir / 'ndvi_mangroves_before_after_combined_panel.png'

fig = plt.figure(figsize=(15, 4.8), constrained_layout=True)
gs = GridSpec(1, 3, figure=fig, width_ratios=[1.25, 1.25, 1.0])

# Panel 1: overlaid before/after histogram
ax1 = fig.add_subplot(gs[0, 0])
low = min(np.percentile(ndvi_before_m, 0.5), np.percentile(ndvi_after_m, 0.5))
high = max(np.percentile(ndvi_before_m, 99.5), np.percentile(ndvi_after_m, 99.5))
bins = np.linspace(low, high, 90)
ax1.hist(ndvi_before_m, bins=bins, alpha=0.50, density=True, label='Before')
ax1.hist(ndvi_after_m, bins=bins, alpha=0.50, density=True, label='After')
ax1.set_title('Mangrove NDVI: Before vs After')
ax1.set_xlabel('NDVI')
ax1.set_ylabel('Density')
ax1.legend(loc='upper left')

# Panel 2: delta histogram
ax2 = fig.add_subplot(gs[0, 1])
q_low, q_high = np.percentile(ndvi_delta_paired, [0.5, 99.5])
delta_bins = np.linspace(q_low, q_high, 90)
ax2.hist(ndvi_delta_paired, bins=delta_bins, alpha=0.8, color='#3366cc', density=True)
ax2.axvline(0, color='black', linestyle='--', linewidth=1)
ax2.set_title('NDVI Change (After - Before)')
ax2.set_xlabel('NDVI change')
ax2.set_ylabel('Density')

# Panel 3: compact summary table
ax3 = fig.add_subplot(gs[0, 2])
ax3.axis('off')

summary_small = pd.DataFrame({
    'Metric': ['n', 'mean', 'median', 'p25', 'p75'],
    'Before': [
        len(ndvi_before_m),
        float(np.mean(ndvi_before_m)),
        float(np.median(ndvi_before_m)),
        float(np.percentile(ndvi_before_m, 25)),
        float(np.percentile(ndvi_before_m, 75)),
    ],
    'After': [
        len(ndvi_after_m),
        float(np.mean(ndvi_after_m)),
        float(np.median(ndvi_after_m)),
        float(np.percentile(ndvi_after_m, 25)),
        float(np.percentile(ndvi_after_m, 75)),
    ],
    'Delta (paired)': [
        len(ndvi_delta_paired),
        float(np.mean(ndvi_delta_paired)),
        float(np.median(ndvi_delta_paired)),
        float(np.percentile(ndvi_delta_paired, 25)),
        float(np.percentile(ndvi_delta_paired, 75)),
    ]
})

summary_show = summary_small.copy()
for col in ['Before', 'After', 'Delta (paired)']:
    summary_show.loc[summary_show['Metric'] == 'n', col] = f"{int(summary_show.loc[summary_show['Metric'] == 'n', col].values[0]):,}"
    for m in ['mean', 'median', 'p25', 'p75']:
        v = float(summary_show.loc[summary_show['Metric'] == m, col].values[0])
        summary_show.loc[summary_show['Metric'] == m, col] = f"{v:.4f}"

table = ax3.table(
    cellText=summary_show.values,
    colLabels=summary_show.columns,
    loc='center',
    cellLoc='center',
)
table.auto_set_font_size(False)
table.set_fontsize(9)
table.scale(1.1, 1.3)
ax3.set_title('Summary')

fig.suptitle('FN Mangrove NDVI Before vs After Hurricane Melissa', fontsize=12)
if SAVE_PNGS:
    fig.savefig(combined_png, dpi=300)
    print('Saved:', combined_png)
else:
    print('PNG export skipped (SAVE_PNGS=False):', combined_png)
plt.show()

In [ ]:
# Geospatial map: NDVI change direction inside FN mangroves
from matplotlib.colors import ListedColormap, BoundaryNorm
import matplotlib.patches as mpatches

with rasterio.open(ndvi_before_path) as src_b, rasterio.open(ndvi_after_path) as src_a:
    arr_before_full = src_b.read(1)
    arr_after_full = src_a.read(1)
    bnds = src_b.bounds

    valid_before_full = np.isfinite(arr_before_full) & (arr_before_full >= -1.0) & (arr_before_full <= 1.0)
    valid_after_full = np.isfinite(arr_after_full) & (arr_after_full >= -1.0) & (arr_after_full <= 1.0)

    if src_b.nodata is not None and np.isfinite(src_b.nodata):
        valid_before_full &= arr_before_full != src_b.nodata
    if src_a.nodata is not None and np.isfinite(src_a.nodata):
        valid_after_full &= arr_after_full != src_a.nodata

    valid_paired_full = valid_before_full & valid_after_full & mangrove_mask
    delta_full = arr_after_full - arr_before_full

# Classification: -1 down, 0 same (exactly zero), +1 up; NaN outside paired mangrove pixels
cls = np.full(delta_full.shape, np.nan, dtype='float32')
cls[valid_paired_full & (delta_full < 0.0)] = -1.0
cls[valid_paired_full & (delta_full == 0.0)] = 0.0
cls[valid_paired_full & (delta_full > 0.0)] = 1.0

# Counts for caption
n_down = int(np.nansum(cls == -1.0))
n_same = int(np.nansum(cls == 0.0))
n_up = int(np.nansum(cls == 1.0))
n_total = n_down + n_same + n_up

# Color map for categories
cmap = ListedColormap(['#d73027', '#bdbdbd', '#1a9850'])  # red, grey, green
norm = BoundaryNorm([-1.5, -0.5, 0.5, 1.5], cmap.N)

# Bigger map canvas
fig, ax = plt.subplots(figsize=(12.5, 9.5), constrained_layout=True)
ax.imshow(
    cls,
    cmap=cmap,
    norm=norm,
    extent=[bnds.left, bnds.right, bnds.bottom, bnds.top],
    origin='upper',
    interpolation='nearest'
)

# Optional: outline mangrove polygons for context
mangroves.boundary.plot(ax=ax, color='black', linewidth=0.15, alpha=0.45)

ax.set_title('FN Mangroves NDVI Change Direction (After - Before)\nSame = exact zero change', fontsize=12)
ax.set_xlabel('Easting (m, EPSG:3448)')
ax.set_ylabel('Northing (m, EPSG:3448)')
ax.set_aspect('equal')

# Small square legend instead of colorbar
legend_handles = [
    mpatches.Patch(facecolor='#d73027', edgecolor='none', label='NDVI down'),
    mpatches.Patch(facecolor='#bdbdbd', edgecolor='none', label='Same'),
    mpatches.Patch(facecolor='#1a9850', edgecolor='none', label='NDVI up'),
]
ax.legend(
    handles=legend_handles,
    loc='upper right',
    frameon=True,
    framealpha=0.92,
    fontsize=9,
    handlelength=0.9,
    handleheight=0.9,
    borderpad=0.35,
    labelspacing=0.3,
)

ax.text(
    0.01,
    0.01,
    f'n={n_total:,}  down={100*n_down/n_total:.1f}%  same={100*n_same/n_total:.1f}%  up={100*n_up/n_total:.1f}%',
    transform=ax.transAxes,
    fontsize=9,
    ha='left',
    va='bottom',
    bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.85, edgecolor='none')
)

map_png = output_dir / 'ndvi_mangroves_change_direction_map_strict_sign_nearest.png'
if SAVE_PNGS:
    fig.savefig(map_png, dpi=320)
    print('Saved:', map_png)
else:
    print('PNG export skipped (SAVE_PNGS=False):', map_png)
plt.show()

In [ ]:
# Substantial change analysis (relative and absolute thresholds)
# Colleague suggestion: focus on substantial changes to reduce noise.

REL_THRESHOLD = 0.10          # 10% relative change threshold
REL_BASELINE_MIN = 0.20       # only compute relative % where baseline NDVI >= this value
ABS_DELTA_THRESHOLD = 0.05    # absolute NDVI-unit threshold (noise-robust comparator)

# Relative change only where baseline is high enough to avoid unstable percentages
eligible_rel = ndvi_before_paired >= REL_BASELINE_MIN
rel_change = np.full_like(ndvi_delta_paired, np.nan, dtype='float32')
rel_change[eligible_rel] = ndvi_delta_paired[eligible_rel] / ndvi_before_paired[eligible_rel]

substantial_rel = np.abs(rel_change) > REL_THRESHOLD
substantial_abs = np.abs(ndvi_delta_paired) > ABS_DELTA_THRESHOLD

summary_substantial = pd.DataFrame([
    {
        'definition': f'Relative: abs((after-before)/before) > {REL_THRESHOLD:.0%}',
        'baseline_condition': f'before >= {REL_BASELINE_MIN:.2f}',
        'n_eligible': int(np.sum(eligible_rel)),
        'n_substantial': int(np.nansum(substantial_rel)),
        'pct_substantial_of_eligible': float(100.0 * np.nansum(substantial_rel) / max(np.sum(eligible_rel), 1)),
        'pct_substantial_of_all_paired': float(100.0 * np.nansum(substantial_rel) / ndvi_delta_paired.size),
        'pct_substantial_decline': float(100.0 * np.nansum((rel_change < -REL_THRESHOLD)) / max(np.sum(eligible_rel), 1)),
        'pct_substantial_improve': float(100.0 * np.nansum((rel_change > REL_THRESHOLD)) / max(np.sum(eligible_rel), 1)),
    },
    {
        'definition': f'Absolute: abs(after-before) > {ABS_DELTA_THRESHOLD:.2f}',
        'baseline_condition': 'none',
        'n_eligible': int(ndvi_delta_paired.size),
        'n_substantial': int(np.sum(substantial_abs)),
        'pct_substantial_of_eligible': float(100.0 * np.sum(substantial_abs) / ndvi_delta_paired.size),
        'pct_substantial_of_all_paired': float(100.0 * np.sum(substantial_abs) / ndvi_delta_paired.size),
        'pct_substantial_decline': float(100.0 * np.sum(ndvi_delta_paired < -ABS_DELTA_THRESHOLD) / ndvi_delta_paired.size),
        'pct_substantial_improve': float(100.0 * np.sum(ndvi_delta_paired > ABS_DELTA_THRESHOLD) / ndvi_delta_paired.size),
    }
]).round(3)

summary_substantial


In [ ]:
# Geospatial map: substantial relative NDVI change in mangroves
# Red = NDVI decreased by more than 10%; Green = increased by more than 10%.
from matplotlib.colors import ListedColormap, BoundaryNorm
import matplotlib.patches as mpatches

# Build full-size relative-change array only where baseline is high enough and paired-valid
eligible_rel_full = valid_paired_m & (arr_before >= REL_BASELINE_MIN)
rel_change_full = np.full(arr_before.shape, np.nan, dtype='float32')
rel_change_full[eligible_rel_full] = (arr_after[eligible_rel_full] - arr_before[eligible_rel_full]) / arr_before[eligible_rel_full]

# Classify: -1 = substantial decline, 0 = not substantial / not eligible, +1 = substantial improvement
cls_rel = np.full(arr_before.shape, np.nan, dtype='float32')
cls_rel[valid_paired_m] = 0.0
cls_rel[eligible_rel_full & (rel_change_full < -REL_THRESHOLD)] = -1.0
cls_rel[eligible_rel_full & (rel_change_full > REL_THRESHOLD)] = 1.0

n_dec = int(np.nansum(cls_rel == -1.0))
n_mid = int(np.nansum(cls_rel == 0.0))
n_inc = int(np.nansum(cls_rel == 1.0))
n_tot = n_dec + n_mid + n_inc

cmap = ListedColormap(['#d73027', '#d9d9d9', '#1a9850'])  # red, grey, green
cmap.set_bad(color='white', alpha=1.0)
norm = BoundaryNorm([-1.5, -0.5, 0.5, 1.5], cmap.N)

fig, ax = plt.subplots(figsize=(12.5, 9.5), constrained_layout=True)
ax.imshow(
    cls_rel,
    cmap=cmap,
    norm=norm,
    extent=[before_meta['bounds'].left, before_meta['bounds'].right, before_meta['bounds'].bottom, before_meta['bounds'].top],
    origin='upper',
    interpolation='nearest',
)

# Outline mangroves for context
mangroves.boundary.plot(ax=ax, color='black', linewidth=0.12, alpha=0.4)

ax.set_title(
    f'FN Mangroves: Relative NDVI Substantial Change (>10%)\nBaseline condition: NDVI before >= {REL_BASELINE_MIN:.2f}',
    fontsize=12,
)
ax.set_xlabel('Easting (m, EPSG:3448)')
ax.set_ylabel('Northing (m, EPSG:3448)')
ax.set_aspect('equal')

legend_handles = [
    mpatches.Patch(facecolor='#d73027', edgecolor='none', label='NDVI decrease > 10%'),
    mpatches.Patch(facecolor='#d9d9d9', edgecolor='none', label='Not substantial / low baseline'),
    mpatches.Patch(facecolor='#1a9850', edgecolor='none', label='NDVI increase > 10%'),
]
ax.legend(
    handles=legend_handles,
    loc='upper right',
    frameon=True,
    framealpha=0.92,
    fontsize=9,
    handlelength=0.9,
    handleheight=0.9,
    borderpad=0.35,
    labelspacing=0.3,
)

ax.text(
    0.01,
    0.01,
    f'n={n_tot:,}  dec={100*n_dec/max(n_tot,1):.1f}%  other={100*n_mid/max(n_tot,1):.1f}%  inc={100*n_inc/max(n_tot,1):.1f}%',
    transform=ax.transAxes,
    fontsize=9,
    ha='left',
    va='bottom',
    bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.85, edgecolor='none'),
)

rel_map_png = output_dir / 'ndvi_mangroves_relative_substantial_change_gt10_map.png'
if SAVE_PNGS:
    fig.savefig(rel_map_png, dpi=320)
    print('Saved:', rel_map_png)
else:
    print('PNG export skipped (SAVE_PNGS=False):', rel_map_png)
plt.show()


In [ ]:
# Optional export: substantial-change summary
substantial_csv = output_dir / 'ndvi_mangroves_substantial_change_summary.csv'
substantial_map_counts_csv = output_dir / 'ndvi_mangroves_relative_substantial_change_gt10_map_counts.csv'

summary_substantial.to_csv(substantial_csv, index=False)
pd.DataFrame([
    {
        'n_total_valid_paired_mangrove': int(n_tot),
        'n_decrease_gt_10pct': int(n_dec),
        'n_increase_gt_10pct': int(n_inc),
        'n_other_not_substantial_or_low_baseline': int(n_mid),
        'pct_decrease_gt_10pct': float(100.0 * n_dec / max(n_tot, 1)),
        'pct_increase_gt_10pct': float(100.0 * n_inc / max(n_tot, 1)),
        'pct_other': float(100.0 * n_mid / max(n_tot, 1)),
        'relative_threshold': REL_THRESHOLD,
        'relative_baseline_min': REL_BASELINE_MIN,
    }
]).to_csv(substantial_map_counts_csv, index=False)

print('Saved:', substantial_csv)
print('Saved:', substantial_map_counts_csv)
